# Análise de lacunas da pesquisa em PLN na USP

Teses e dissertações da Biblioteca Digital da USP coletadas com o Octoparse.

**Etapas de PLN:**
1. Pré-processamento (limpeza, tokenização, *stopwords*)
2. Lematização e classes gramaticais (spaCy)
3. Sintagmas nominais
4. Termos frequentes e termos emergentes
5. Classificação de tarefas e métodos por dicionário (`PhraseMatcher`)
6. TF-IDF e modelagem de tópicos (NMF)
7. Análise de lacunas (cruzamentos)
8. Validação com anotação manual

## 0. Configuração e bibliotecas

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from spacy.matcher import PhraseMatcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

ARQUIVO = Path('../DATA/Biblioteca_de_Teses_Scraping_xml.xlsx')   # exportação do Octoparse
NIVEIS = ['Dissertação de Mestrado', 'Tese de Doutorado', 'Livre-docência']  # tire algum para excluí-lo
N_TOPICOS = 6

PASTA = Path('../DATA')
(PASTA / 'resultados').mkdir(exist_ok=True)

try:
    nlp = spacy.load('pt_core_news_lg')
except OSError:
    nlp = spacy.load('pt_core_news_sm')   # modelo menor, se o grande não estiver instalado

## 1. Leitura e limpeza dos dados

In [ ]:
if ARQUIVO.exists():
    bruto = pd.read_excel(ARQUIVO, sheet_name=0)
    df = pd.DataFrame({
        'doi': bruto['DOI'],
        'titulo': bruto['Título_Port'],
        'nivel': bruto['Documento'],
        'unidade': bruto['Unidade_USP'],
        'area': bruto['Area'],
        'ano': pd.to_datetime(bruto['Data_Defesa']).dt.year,
        'palavras_chave': bruto['Palavras_chave_Port'],
        'resumo': bruto['Resumo_Port'],
    })
    # A coluna Autor não entra: não é usada na análise e é dado pessoal
else:
    print('Arquivo do Octoparse não encontrado: usando a amostra sintética')
    df = pd.read_csv(PASTA / 'teses_sinteticas.csv').rename(
        columns={'area_concentracao': 'area', 'resumo_pt': 'resumo'})
    df['doi'] = df['id_trabalho']

# Limpeza: espaços e quebras de linha repetidos, duplicatas e níveis fora do recorte
for col in ['titulo', 'palavras_chave', 'resumo']:
    df[col] = df[col].fillna('').str.split().str.join(' ')
df = df[df['nivel'].isin(NIVEIS)].drop_duplicates('doi').reset_index(drop=True)
df['url'] = 'https://doi.org/' + df['doi']

# Alguns trabalhos têm o resumo em inglês no campo de português: conta palavras típicas de cada idioma
ingles = df['resumo'].str.lower().str.count(r'\b(?:the|and|of|is|this)\b')
portugues = df['resumo'].str.lower().str.count(r'\b(?:o|a|de|que|é|um|uma)\b')
df['idioma'] = (ingles > portugues).map({True: 'en', False: 'pt'})
print('Resumos em inglês:', (df['idioma'] == 'en').sum())

# Base limpa e sem dados pessoais (pode ser publicada, com crédito à fonte)
df.to_csv(PASTA / 'teses_usp_pln.csv', index=False, encoding='utf-8')
print(len(df), 'trabalhos')
df.head()

### Perfil da produção

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
pd.crosstab(df['ano'], df['nivel']).plot.bar(stacked=True, ax=ax[0], title='Trabalhos por ano')
df['unidade'].value_counts().sort_values().plot.barh(ax=ax[1], title='Trabalhos por unidade')
plt.tight_layout()

## 2. Pré-processamento, lematização e classes gramaticais (spaCy)

Para cada resumo, o spaCy separa as palavras (tokenização), identifica a classe gramatical (substantivo, verbo...) e a forma básica (lema). Ficam só substantivos, adjetivos e nomes próprios, sem *stopwords*.

O modelo é de português, então os resumos em inglês ficam fora das etapas 2 a 4 e 6 (termos e tópicos). Eles continuam na classificação da etapa 5, porque o dicionário tem termos nos dois idiomas.

In [ ]:
# Palavras comuns a quase todo resumo acadêmico, que não indicam tema
STOP_ACADEMICAS = {'trabalho', 'estudo', 'objetivo', 'resultado', 'análise', 'método', 'pesquisa', 'dado',
                   'abordagem', 'proposta', 'dissertação', 'tese', 'português', 'brasileiro', 'texto', 'linguagem'}

docs = list(nlp.pipe(df['resumo']))

df['lemas'] = [[t.lemma_.lower() for t in doc
                if t.pos_ in ('NOUN', 'ADJ', 'PROPN') and t.is_alpha and not t.is_stop
                and len(t) > 2 and t.lemma_.lower() not in STOP_ACADEMICAS]
               if idioma == 'pt' else []
               for doc, idioma in zip(docs, df['idioma'])]

# Exemplo do que o spaCy faz em uma frase
pd.DataFrame([(t.text, t.lemma_, t.pos_) for t in docs[0][:15]], columns=['palavra', 'lema', 'classe'])

## 3. Sintagmas nominais

Termos compostos (ex.: *"reconhecimento de entidades nomeadas"*), que representam melhor os conceitos da área do que palavras soltas.

In [ ]:
def sintagmas(doc):
    saida = []
    for sn in doc.noun_chunks:
        palavras = [t.text.lower() for t in sn if not (t.is_stop and t.i == sn.start)]  # tira o artigo inicial
        if len(palavras) >= 2:
            saida.append(' '.join(palavras))
    return saida

df['sintagmas'] = [sintagmas(doc) if idioma == 'pt' else [] for doc, idioma in zip(docs, df['idioma'])]

## 4. Termos frequentes e termos emergentes

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df['lemas'].explode().value_counts().head(20).sort_values().plot.barh(ax=ax[0], title='Termos mais frequentes')
df['sintagmas'].explode().value_counts().head(20).sort_values().plot.barh(ax=ax[1], title='Sintagmas mais frequentes')
plt.tight_layout()

In [ ]:
# Compara a frequência relativa de cada termo antes e depois do ano mediano
meio = int(df['ano'].median())
df['periodo'] = df['ano'].apply(lambda a: f'até {meio}' if a <= meio else f'depois de {meio}')

freq = df.explode('lemas').groupby(['lemas', 'periodo']).size().unstack(fill_value=0)
freq = freq / freq.sum()
freq['variacao'] = freq[f'depois de {meio}'] - freq[f'até {meio}']

print('Termos emergentes:', ', '.join(freq['variacao'].nlargest(15).index))
print('Termos em queda:  ', ', '.join(freq['variacao'].nsmallest(15).index))

## 5. Classificação de tarefas e métodos (dicionário + `PhraseMatcher`)

Cada trabalho recebe as tarefas e os métodos cujas expressões aparecem no título, nas palavras-chave ou no resumo. O dicionário mistura português e inglês, porque os resumos usam termos como *word embeddings* e *transformers*.

In [ ]:
TAREFAS = {
    'sentimentos e opinião': ['análise de sentimento', 'análise de sentimentos', 'sentiment analysis',
                              'mineração de opinião', 'polaridade', 'posicionamento', 'emoção', 'emoções'],
    'linguagem ofensiva e ironia': ['discurso de ódio', 'hate speech', 'ironia', 'sarcasmo', 'desinformação',
                                    'notícias falsas', 'fake news'],
    'entidades e extração de informação': ['entidades nomeadas', 'reconhecimento de entidades', 'named entity',
                                           'extração de informação', 'extração de informações'],
    'classificação de textos': ['classificação de textos', 'classificação de texto', 'classificação automática',
                                'classificação de documentos', 'text classification', 'classificação multirrótulo'],
    'recuperação de informação': ['recuperação de informação', 'recuperação de informações',
                                  'information retrieval', 'busca semântica', 'pesquisa de jurisprudência'],
    'sumarização': ['sumarização', 'summarization', 'sumário automático'],
    'geração de texto': ['geração de texto', 'geração automática', 'text generation', 'geração de linguagem'],
    'tradução automática': ['tradução automática', 'tradutor automático', 'machine translation'],
    'perguntas, diálogo e interfaces': ['perguntas e respostas', 'question answering', 'chatbot',
                                        'agente conversacional', 'interfaces em linguagem natural', 'sparql'],
    'semântica e desambiguação': ['desambiguação', 'implicação textual', 'similaridade semântica', 'ontologia',
                                  'ontologias'],
    'morfologia e sintaxe': ['análise sintática', 'analisador sintático', 'parser', 'parsing', 'treebank',
                             'etiquetagem', 'etiquetador', 'análise morfossintática', 'analisador morfológico'],
    'avaliação e simplificação de textos': ['simplificação textual', 'legibilidade', 'readability',
                                            'avaliação automática de redações', 'correção automatizada',
                                            'qualidade de escrita'],
    'modelagem de tópicos': ['modelagem de tópicos', 'modelo de tópicos', 'topic modeling'],
    'fala': ['reconhecimento de fala', 'síntese de fala', 'speech recognition', 'prosódia'],
    'recursos linguísticos (corpus, léxico)': ['construção de um corpus', 'construção de corpus', 'corpus anotado',
                                               'construção de um léxico', 'terminologia', 'linguística de corpus'],
}

METODOS = {   # em ordem cronológica aproximada
    '1. regras e léxicos': ['baseado em regras', 'baseada em regras', 'rule-based', 'gramática', 'gramáticas',
                            'baseado em léxico', 'autômatos', 'lógica'],
    '2. estatística e aprendizado de máquina': ['aprendizado de máquina', 'machine learning', 'aprendizado computacional',
                                                 'aprendizagem de máquina', 'svm', 'naive bayes', 'random forest',
                                                 'regressão logística', 'crf', 'cadeias de markov', 'tf-idf'],
    '3. redes neurais e embeddings': ['rede neural', 'redes neurais', 'neural network', 'lstm', 'cnn', 'word2vec',
                                      'word embeddings', 'embeddings', 'aprendizado profundo', 'deep learning',
                                      'representação vetorial', 'representações vetoriais'],
    '4. transformers e BERT': ['transformer', 'transformers', 'bert', 'bertimbau', 'roberta', 'atenção neural',
                               'ajuste fino', 'fine-tuning'],
    '5. LLMs': ['llm', 'llms', 'grandes modelos de linguagem', 'large language model', 'gpt', 'chatgpt',
                'llama', 'prompt', 'prompts', 'zero-shot'],
}

def classificar(dicionario):
    matcher = PhraseMatcher(nlp.vocab, attr='LOWER')
    for rotulo, termos in dicionario.items():
        matcher.add(rotulo, list(nlp.tokenizer.pipe(termos)))
    textos = nlp.pipe(df['titulo'] + '. ' + df['palavras_chave'] + '. ' + df['resumo'])
    return [sorted({nlp.vocab.strings[m] for m, _, _ in matcher(doc)}) for doc in textos]

df['tarefas'] = classificar(TAREFAS)
df['metodos'] = classificar(METODOS)

print('Sem tarefa identificada:', (df['tarefas'].str.len() == 0).sum(), 'de', len(df))
df[['ano', 'titulo', 'tarefas', 'metodos']].head(10)

In [ ]:
tarefas = df.explode('tarefas', ignore_index=True).dropna(subset=['tarefas'])
metodos = df.explode('metodos', ignore_index=True).dropna(subset=['metodos'])

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
tarefas['tarefas'].value_counts().sort_values().plot.barh(ax=ax[0], title='Tarefas de PLN')
pd.crosstab(metodos['ano'], metodos['metodos']).plot.bar(stacked=True, ax=ax[1], title='Métodos por ano')
plt.tight_layout()

## 6. TF-IDF e modelagem de tópicos (NMF)

Agrupa os trabalhos por temas **sem usar o dicionário**. Compare com as tarefas da etapa 5: tópicos que não correspondem a nenhuma tarefa indicam temas que o dicionário não previu.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9)
X = tfidf.fit_transform(df['lemas'].str.join(' '))

nmf = NMF(n_components=N_TOPICOS, random_state=42, init='nndsvda', max_iter=500)
pesos = nmf.fit_transform(X)
vocab = tfidf.get_feature_names_out()

df['topico'] = [f'T{i}' if idioma == 'pt' else None for i, idioma in zip(pesos.argmax(axis=1), df['idioma'])]
nomes = [f'T{i}' for i in range(N_TOPICOS)]
topicos = pd.DataFrame({
    'termos principais': [', '.join(vocab[j] for j in comp.argsort()[::-1][:8]) for comp in nmf.components_],
    'nº de trabalhos': df['topico'].value_counts().reindex(nomes, fill_value=0).values,
}, index=nomes)
topicos

## 7. Análise de lacunas

Combinações com **zero ou poucos trabalhos** são as lacunas: tarefas pouco estudadas, tarefas ainda não estudadas com métodos recentes e áreas que usam PLN só de forma limitada.

In [ ]:
def mapa_de_calor(tabela, titulo):
    fig, ax = plt.subplots(figsize=(1 + 0.9 * tabela.shape[1], 0.45 * tabela.shape[0] + 1.5))
    im = ax.imshow(tabela.values, cmap='Blues')
    ax.set_xticks(range(tabela.shape[1]), tabela.columns, rotation=45, ha='right')
    ax.set_yticks(range(tabela.shape[0]), tabela.index)
    for i in range(tabela.shape[0]):
        for j in range(tabela.shape[1]):
            ax.text(j, i, tabela.iat[i, j], ha='center', va='center', fontsize=8)
    ax.set_title(titulo)
    plt.tight_layout()

tm = tarefas.explode('metodos', ignore_index=True).dropna(subset=['metodos'])
tarefa_metodo = pd.crosstab(tm['tarefas'], tm['metodos'])
mapa_de_calor(tarefa_metodo, 'Tarefa × método')

In [ ]:
unidade_tarefa = pd.crosstab(tarefas['unidade'], tarefas['tarefas'])
mapa_de_calor(unidade_tarefa, 'Unidade × tarefa')

In [ ]:
tarefa_periodo = pd.crosstab(tarefas['tarefas'], tarefas['periodo'])
tarefa_periodo['variação'] = tarefa_periodo[f'depois de {meio}'] - tarefa_periodo[f'até {meio}']
tarefa_periodo.sort_values('variação')

In [ ]:
# Lista automática de lacunas
recentes = [m for m in tarefa_metodo.columns if m.startswith(('4.', '5.'))]
lacunas = pd.DataFrame({
    'nº de trabalhos': tarefas['tarefas'].value_counts(),
    'com transformers ou LLMs': tarefa_metodo[recentes].sum(axis=1) if recentes else 0,
    'nº de unidades': tarefas.groupby('tarefas')['unidade'].nunique(),
}).reindex(TAREFAS).fillna(0).astype(int)
lacunas['lacuna'] = (lacunas['nº de trabalhos'] <= 3) | (lacunas['com transformers ou LLMs'] == 0)
lacunas.sort_values(['lacuna', 'nº de trabalhos'], ascending=[False, True])

In [ ]:
# Salva as tabelas para o README e a apresentação
df.drop(columns=['lemas', 'sintagmas', 'resumo']).to_csv(PASTA / 'resultados' / 'trabalhos_classificados.csv', index=False)
topicos.to_csv(PASTA / 'resultados' / 'topicos.csv')
lacunas.to_csv(PASTA / 'resultados' / 'lacunas.csv')

## 8. Validação com anotação manual

1. A célula abaixo cria `DATA/anotacao_manual.csv` com 40 trabalhos sorteados.
2. Leia cada resumo e preencha `tarefa_manual` e `metodo_manual` com os rótulos dos dicionários (vários separados por `;`).
3. Rode a célula seguinte: ela mostra a **precisão** (quanto do que o dicionário marcou está certo) e a **cobertura** (quanto do que existe o dicionário encontrou).

In [ ]:
ANOTACAO = PASTA / 'anotacao_manual.csv'
if not ANOTACAO.exists():
    amostra = df.sample(min(40, len(df)), random_state=42)[['doi', 'titulo', 'resumo']]
    amostra.assign(tarefa_manual='', metodo_manual='').to_csv(ANOTACAO, index=False)
    print('Planilha criada: preencha e rode a próxima célula')

In [ ]:
anot = pd.read_csv(ANOTACAO).merge(df[['doi', 'tarefas', 'metodos']], on='doi')

def avaliar(col_auto, col_manual, rotulos):
    anotados = anot.dropna(subset=[col_manual])
    if anotados.empty:
        return 'Nenhum trabalho anotado ainda.'
    manual = anotados[col_manual].str.split(';').apply(lambda l: {r.strip() for r in l})
    auto = anotados[col_auto].apply(set)
    linhas = []
    for r in rotulos:
        vp = sum(r in a and r in m for a, m in zip(auto, manual))
        fp = sum(r in a and r not in m for a, m in zip(auto, manual))
        fn = sum(r not in a and r in m for a, m in zip(auto, manual))
        if vp + fp + fn:
            linhas.append({'rótulo': r,
                           'precisão': round(vp / (vp + fp), 2) if vp + fp else None,
                           'cobertura': round(vp / (vp + fn), 2) if vp + fn else None})
    return pd.DataFrame(linhas)

display(avaliar('tarefas', 'tarefa_manual', TAREFAS))
display(avaliar('metodos', 'metodo_manual', METODOS))